# importy

In [24]:
from dotenv import load_dotenv
import pandas as pd
import numpy as np
from tqdm import tqdm
from datetime import datetime
import json

import requests
import os
import pickle

# dane historyczne

In [25]:
load_dotenv()
RAPIDAPI_KEY = os.environ["RAPIDAPI_KEY"]

In [26]:
url = "https://meteostat.p.rapidapi.com/stations/nearby"

params = {"lat":"49.29543136293499","lon":"19.94964974232761"}

headers = {
	"x-rapidapi-key": RAPIDAPI_KEY,
	"x-rapidapi-host": "meteostat.p.rapidapi.com",
	"Content-Type": "application/json"
}

response = requests.get(url, headers=headers, params=params)

response.json()

{'meta': {'generated': '2026-03-21 09:28:08'},
 'data': [{'id': '12625', 'name': {'en': 'Zakopane'}, 'distance': 508.6},
  {'id': '12650', 'name': {'en': 'Kasprowy Wierch'}, 'distance': 7327.6},
  {'id': 'EPNT0', 'name': {'en': 'Nowy Targ / Nowytarg'}, 'distance': 19986.0},
  {'id': '11934', 'name': {'en': 'Poprad / Tatry'}, 'distance': 33518.1},
  {'id': '12660', 'name': {'en': 'Nowy Sacz'}, 'distance': 64942.9},
  {'id': '12566', 'name': {'en': 'Krakow'}, 'distance': 88266.0},
  {'id': '12600', 'name': {'en': 'Bielsko-Biala'}, 'distance': 88552.5},
  {'id': '11903', 'name': {'en': 'Sliac'}, 'distance': 92505.1},
  {'id': '12766', 'name': {'en': 'Josvafo'}, 'distance': 99878.6}]}

In [27]:
stations_dict = {"12625": "Zakopane",
                 "12650": "Kasprowy Wierch"}

In [28]:
dfs = []
for station_id in stations_dict:
    url = "https://meteostat.p.rapidapi.com/stations/daily"

    querystring = {"station": station_id, "start":"2020-01-01", "end":"2025-12-31"}
    
    headers = {
    	"x-rapidapi-key": RAPIDAPI_KEY,
    	"x-rapidapi-host": "meteostat.p.rapidapi.com",
    	"Content-Type": "application/json"
    }
    
    response = requests.get(url, headers=headers, params=querystring)
    data = response.json()["data"]
    df = pd.DataFrame(data)
    df["station_name"] = stations_dict[station_id]
    dfs.append(df)

In [29]:
df_api_data = pd.concat(dfs, axis=0).reset_index(drop=True)

In [30]:
list(df_api_data.columns)

['date',
 'tavg',
 'tmin',
 'tmax',
 'prcp',
 'snow',
 'wdir',
 'wspd',
 'wpgt',
 'pres',
 'tsun',
 'station_name']

In [31]:
# Parameter	Description	Type
# date	The date string (YYYY-MM-DD)	String
# tavg	The average air temperature in °C	Float
# tmin	The minimum air temperature in °C	Float
# tmax	The maximum air temperature in °C	Float
# prcp	The daily precipitation total in mm	Float
# snow	The maximum snow depth in mm	Integer
# wdir	The average wind direction in degrees (°)	Integer
# wspd	The average wind speed in km/h	Float
# wpgt	The peak wind gust in km/h	Float
# pres	The average sea-level air pressure in hPa	Float
# tsun	The daily sunshine total in minutes (m)	Integer

In [32]:
columns_dict = {
    'tavg': "avg_temp",
    'tmin': "min_temp",
    'tmax': "max_temp",
    'prcp': "precipitation_total_mm",
    'wdir': "wind_direction",
    'wspd': "average_wind_speed",
    'wpgt': "max_wind_speed",
    'pres': "pressure_hpa",
    'tsun': "sunshine_total_min"
}

In [33]:
df_api_data = df_api_data.rename(columns_dict, axis=1)

In [34]:
df_api_data

,date,avg_temp,min_temp,max_temp,precipitation_total_mm,snow,wind_direction,average_wind_speed,max_wind_speed,pressure_hpa,sunshine_total_min,station_name
0,2020-01-01 00:00:00,-1.7,-4.6,0.1,NaN,None,None,6.8,18.5,1033.2,None,Zakopane
1,2020-01-02 00:00:00,-1.7,-5.4,5.6,NaN,None,None,5.6,13.0,1031.1,None,Zakopane
2,2020-01-03 00:00:00,-1.4,-6.0,5.8,NaN,None,None,4.6,22.2,1025.6,None,Zakopane
3,2020-01-04 00:00:00,0.7,-2.4,4.5,NaN,None,None,13.2,33.3,1020.9,None,Zakopane
4,2020-01-05 00:00:00,-4.1,-5.1,-2.5,NaN,None,None,8.9,31.5,1029.3,None,Zakopane
...,...,...,...,...,...,...,...,...,...,...,...,...
4379,2025-12-27 00:00:00,-5.6,-10.3,1.3,0.0,None,None,12.2,None,1021.8,None,Kasprowy Wierch
4380,2025-12-28 00:00:00,-12.9,-14.6,-10.6,1.1,None,None,23.7,None,1022.6,None,Kasprowy Wierch
4381,2025-12-29 00:00:00,-11.5,-14.7,-7.8,0.0,None,None,17.7,None,1016.0,None,Kasprowy Wierch
4382,2025-12-30 00:00:00,-14.7,-17.1,-11.3,4.4,None,None,19.2,None,1015.0,None,Kasprowy Wierch


In [35]:
df_api_data.to_csv("data/historical_for_eda.csv", index=False)

In [36]:
# with open("data/stations_daily.pkl", "wb") as f:
#     pickle.dump(df_api_data, f)

# Dane bieżące

In [37]:
load_dotenv()
OPENWEATHERAPI_KEY = os.environ["OPENWEATHERAPI_KEY"]

In [38]:
latitudes = np.linspace(49.17035070524409, 49.309555578150245, 10)
longitutes = np.linspace(19.76220706945233, 20.125423430066586, 10)

In [39]:
def get_current_weather(lat, lon):
    url = "https://api.openweathermap.org/data/2.5/weather"

    params = {
        "lat": lat,
        "lon": lon,
        "appid": OPENWEATHERAPI_KEY,
        "units": "metric"
    }
    
    response = requests.get(url, params=params)
    return response.json()["main"]

In [40]:
def get_air_pollution(lat, lon):
    url = "http://api.openweathermap.org/data/2.5/air_pollution"

    params = {
        "lat": lat,
        "lon": lon,
        "appid": OPENWEATHERAPI_KEY,
    }
    
    response = requests.get(url, params=params)
    return response.json()['list'][0]["components"]["pm10"]

In [41]:
dfs = []
for lat in tqdm(latitudes):
    for lon in longitutes:
        data = get_current_weather(lat, lon)
        df = pd.DataFrame(data, index=[0])
        df["lat"] = lat
        df["lon"] = lon
        df["pm10"] = get_air_pollution(lat, lon)
        df["download_timestamp"] = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        dfs.append(df)

df_api_openweather = pd.concat(dfs)

100%|███████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:35<00:00,  3.54s/it]


In [42]:
cols_to_drop = ['temp_min', 'temp_max', 'sea_level',  'grnd_level']
df_api_openweather = df_api_openweather.drop(cols_to_drop, axis=1, errors=True)
df_api_openweather = df_api_openweather.reset_index(drop=True)

In [43]:
header_state = not "weather_history.csv" in os.listdir("data")

In [44]:
df_api_openweather.to_csv("data/weather_history.csv", mode="a", header=header_state, index=False)

# Prognoza

In [45]:
def get_forecast(lat, lon):
    url = "http://api.openweathermap.org/data/2.5/forecast"

    params = {
        "lat": lat,
        "lon": lon,
        "appid": OPENWEATHERAPI_KEY,
        "units": "metric"
    }
    
    response = requests.get(url, params=params)
    response_json = response.json()

    temps = [item["main"]["temp"] for item in response_json["list"]]
    tmstps = [item["dt_txt"] for item in response_json["list"]]
    
    return {
        "lat": lat,
        "lon": lon,
        "temperatures": 
            dict(zip(tmstps, temps))
    }

In [46]:
json_filename = f'{datetime.now().strftime("%Y%m%d_%H%M%S")}.json'
forecast_data = []

for lat in tqdm(latitudes):
    for lon in longitutes:
        forecast_data.append(get_forecast(lat, lon))

with open(f"data/json/{json_filename}", "w", encoding="utf-8") as f:
    json.dump(forecast_data, f, indent=2)

100%|███████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:09<00:00,  1.05it/s]
